# Day 34 AM – Part A: Clustering on Iris

K-Means and DBSCAN on Iris without using labels for training.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.decomposition import PCA


In [ ]:
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Shape:", X.shape)
print("Feature names:", feature_names)
print("Class counts:", np.bincount(y))


In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

cross_tab = pd.crosstab(pd.Series(y, name="True Species"), pd.Series(kmeans_labels, name="KMeans Cluster"))
ari = adjusted_rand_score(y, kmeans_labels)
nmi = normalized_mutual_info_score(y, kmeans_labels)

print("Confusion-matrix-like table:")
print(cross_tab)
print("
ARI:", round(ari, 4))
print("NMI:", round(nmi, 4))


In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap="viridis", edgecolor="k")
axes[0].set_title("True Labels")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels, cmap="viridis", edgecolor="k")
axes[1].set_title("K-Means Labels")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")

plt.tight_layout()
plt.savefig("iris_true_vs_kmeans.png", dpi=150)
plt.show()


In [ ]:
dbscan = DBSCAN(eps=0.8, min_samples=5)
db_labels = dbscan.fit_predict(X_scaled)

unique_labels, counts = np.unique(db_labels, return_counts=True)
print("DBSCAN labels and counts:")
for label, count in zip(unique_labels, counts):
    print(label, count)

valid_mask = db_labels != -1
if len(np.unique(db_labels[valid_mask])) > 1:
    db_ari = adjusted_rand_score(y[valid_mask], db_labels[valid_mask])
    print("DBSCAN ARI on non-noise points:", round(db_ari, 4))
else:
    print("DBSCAN did not form enough clusters for ARI.")


## Interpretation

If unsupervised clustering agrees well with known labels, it suggests the natural structure of the feature space already separates the groups.
For Iris, setosa is usually recovered very clearly, while versicolor and virginica overlap more, so some confusion is expected even when clustering works reasonably well.